# MnasNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，MnasNetを用いてCIFAR100データセットの100クラス物体認識を行う．MobileNetV2のInverted Residual構造にSqueeze-and-Excitation（SE）構造を組み合わせたブロックと，モバイル端末上での推論速度を考慮したNeural Architecture Search（NAS）によって段ごとに構成が調整されたネットワーク構造について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100は32×32ピクセルのカラー画像100クラスからなるデータセットです．学習用データには`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Squeeze-and-Excitation（SE）
MnasNetのブロックは，1×1畳み込みによるチャンネル拡張・Depthwise Convolution・1×1畳み込みによるチャンネル削減という，MobileNetV2のInverted Residual構造をベースに，Squeeze-and-Excitation（SE）構造を追加したものです．

SE構造は，チャンネルごとの重要度を特徴マップ自身から算出し，各チャンネルを重み付けし直す仕組みです．まずGlobal Average Poolingで各チャンネルを1つの値に集約し（Squeeze），全結合層・ReLU・全結合層・Sigmoidからなる小さなネットワークでチャンネルごとの重み（0〜1の値）を求めます（Excitation）．最後に，元の特徴マップの各チャンネルにこの重みを掛け合わせることで，重要なチャンネルを強調し，重要度の低いチャンネルを抑制します．

MnasNetは，このSE付きのブロック（MBConv）を用いて，モバイル端末上での推論速度を考慮しながら精度を最大化するように，段ごとの構成（カーネルサイズ，拡張比率，SEの有無など）をNeural Architecture Search（NAS）によって探索したネットワークです．本ノートブックでは，探索処理そのものではなく，探索によって得られた結果（MnasNet-A1相当）の構成をそのまま実装します．

In [ ]:
class SEModule(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        reduced_ch = max(1, channels // reduction)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_ch),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_ch, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.avgpool(x).view(b, c)
        s = self.fc(s).view(b, c, 1, 1)
        return x * s

## SE付きInverted Residual Block（MBConv）
Inverted Residual構造にSE構造を追加したブロックを定義します．SE構造は，Depthwise Convolutionの直後（チャンネル削減を行う1×1畳み込みの前）に挿入します．また，MnasNetでは段ごとにカーネルサイズ（3×3または5×5）や拡張比率$t$，SEの有無が異なるため，これらを引数として受け取れるようにします．最後のProjection（1×1畳み込み）の直後には活性化関数を適用せず（Linear Bottleneck），ショートカット接続は，strideが1かつ入力と出力のチャンネル数が等しい場合のみ行います．

In [ ]:
class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride, expand_ratio, use_se):
        super().__init__()
        hidden_ch = in_ch * expand_ratio
        self.use_res = stride == 1 and in_ch == out_ch

        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hidden_ch, kernel_size=1, bias=False),
                       nn.BatchNorm2d(hidden_ch),
                       nn.ReLU(inplace=True)]
        layers += [
            nn.Conv2d(hidden_ch, hidden_ch, kernel_size=kernel_size, stride=stride,
                       padding=kernel_size // 2, groups=hidden_ch, bias=False),
            nn.BatchNorm2d(hidden_ch),
            nn.ReLU(inplace=True),
        ]
        self.expand_dw = nn.Sequential(*layers)

        self.se = SEModule(hidden_ch) if use_se else nn.Identity()

        # Projection．活性化関数は適用しない（Linear Bottleneck）
        self.project = nn.Sequential(
            nn.Conv2d(hidden_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        out = self.expand_dw(x)
        out = self.se(out)
        out = self.project(out)
        if self.use_res:
            out = out + x
        return out

## ネットワークの定義
MnasNet-A1相当の構成として，段ごとに以下の表のブロックを積み重ねます．各行は，カーネルサイズ$k$，拡張比率$t$，出力チャンネル数$c$，ブロックの繰り返し数$n$，各段最初のブロックのstride $s$，SE構造の有無を表します．

| k | t | c | n | s | SE |
|---|---|---|---|---|---|
| 3 | 1 | 16 | 1 | 1 | - |
| 3 | 6 | 24 | 2 | 1 | - |
| 5 | 3 | 40 | 3 | 2 | ○ |
| 3 | 6 | 80 | 4 | 1 | - |
| 3 | 6 | 112 | 2 | 1 | ○ |
| 5 | 6 | 160 | 3 | 2 | ○ |
| 3 | 6 | 320 | 1 | 1 | - |

最初の3×3畳み込み（stem）で入力画像を32チャンネルに変換したのち，上表のブロックを順に適用します．最後に1×1畳み込みで1280チャンネルに拡張し，Global Average Poolingで空間方向を1×1に集約した後，全結合層でクラス数（100）に変換します．

なお，MnasNetにも，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR100のような小さな入力を対象とした「CIFAR版」があります．オリジナルのMnasNet-A1はstemと複数の段のstride=2によって特徴マップを合計32分の1まで縮小しますが，32×32の入力にそのまま適用すると特徴マップが1×1以下になってしまいます．そのため，本ノートブックでは上表のようにstemのstrideとstride=2となる段数を減らしたCIFAR版を実装します（詳細は本ノートブック末尾の「ImageNet版MnasNetとCIFAR版（本ノートブック）の違い」を参照）．

In [ ]:
def _make_stage(in_ch, out_ch, kernel_size, n_blocks, stride, expand_ratio, use_se):
    layers = [MBConv(in_ch, out_ch, kernel_size, stride, expand_ratio, use_se)]
    for _ in range(n_blocks - 1):
        layers.append(MBConv(out_ch, out_ch, kernel_size, 1, expand_ratio, use_se))
    return nn.Sequential(*layers)


class MnasNet(nn.Module):
    # k, t, c, n, s, se
    cfg = [
        [3, 1, 16, 1, 1, False],
        [3, 6, 24, 2, 1, False],   # 元のImageNet版はs=2だが，CIFAR100 (32x32)向けにs=1へ変更
        [5, 3, 40, 3, 2, True],
        [3, 6, 80, 4, 1, False],   # 元のImageNet版はs=2だが，CIFAR100 (32x32)向けにs=1へ変更
        [3, 6, 112, 2, 1, True],
        [5, 6, 160, 3, 2, True],
        [3, 6, 320, 1, 1, False],
    ]

    def __init__(self, n_class=100):
        super().__init__()
        in_ch = 32
        last_ch = 1280

        self.stem = nn.Sequential(
            nn.Conv2d(3, in_ch, kernel_size=3, stride=1, padding=1, bias=False),  # CIFAR100向けにstride=1
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
        )

        stages = []
        for k, t, c, n, s, se in self.cfg:
            stages.append(_make_stage(in_ch, c, k, n, s, t, se))
            in_ch = c
        self.blocks = nn.Sequential(*stages)

        self.head = nn.Sequential(
            nn.Conv2d(in_ch, last_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(last_ch),
            nn.ReLU(inplace=True),
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(last_ch, n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
MnasNetを作成し，`.to(device)`によりモデルのパラメータを`device`上に配置します．最適化手法にはモーメンタムSGD（学習率0.01，モーメンタム0.9，weight decay 5e-4）を用います．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = MnasNet(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版MnasNetとCIFAR版（本ノートブック）の違い

| | ImageNet版（MnasNet-A1） | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| stemの畳み込みのstride | 2 | 1 |
| ブロック内でstride=2となる段数 | 4段 | 2段（c=40, c=160の段のみ） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

MnasNetもMobileNetV2と同様，ImageNet（224×224入力）向けに設計されているため，32×32のCIFAR100にそのまま適用すると特徴マップが小さくなりすぎてしまいます．そこで本ノートブックでは，stemのstrideを1にし，ブロック内でstride=2となる段数を4段から2段に減らすことで，最終的な特徴マップサイズを8×8としています．

また，本ノートブックで実装しているのはMnasNetの探索によって得られた**結果のネットワーク構造**（MobileNetV2系のMBConvブロックにSE構造を追加し，段ごとにカーネルサイズ・拡張比率・SEの有無を変えたもの）であり，論文の主眼であるモバイル端末上の推論速度を考慮したNeural Architecture Search（NAS）による探索処理そのものは行っていません．段ごとの構成は，MnasNet-A1として報告されている構成を参考にしています．

## 課題

1. SE構造を全てのブロックから取り除いたネットワークを作成し，SEの有無で認識精度やパラメータ数がどのように変化するか比較しましょう．
2. 各段のカーネルサイズ（3×3, 5×5）や拡張比率$t$を変更して，パラメータ数や認識精度への影響を確認しましょう．
3. SEモジュールの`reduction`（削減率）を変更し，認識精度や計算コストへの影響を確認しましょう．